In [2]:
import json
import csv
from pathlib import Path
from collections import defaultdict

# Install BERTScore (run once)
# !pip install bert-score

In [3]:
# The two models to compare
BASE_MODEL = "llama3.1:8b"       # Ground truth answers
COMPARE_MODEL = "mistral:latest"  # Answers to compare

BASE_TAG = BASE_MODEL.replace(":", "_").replace(".", "_").replace("-", "_")
COMPARE_TAG = COMPARE_MODEL.replace(":", "_").replace(".", "_").replace("-", "_")

print(f"[INFO] Base model (ground truth): {BASE_MODEL} -> {BASE_TAG}")
print(f"[INFO] Compare model: {COMPARE_MODEL} -> {COMPARE_TAG}")

[INFO] Base model (ground truth): llama3.1:8b -> llama3_1_8b
[INFO] Compare model: mistral:latest -> mistral_latest


In [4]:
SA_DATASET_NAMES = ["DoS", "Fuzzy", "Gear", "RPM"]

SA_QUESTION_FILES = {
    "DoS":   Path("DoS_sa_qa/questions/dos_sa_questions.json"),
    "Fuzzy": Path("Fuzzy_sa_qa/questions/fuzzy_sa_questions.json"),
    "Gear":  Path("Gear_sa_qa/questions/gear_sa_questions.json"),
    "RPM":   Path("RPM_sa_qa/questions/rpm_sa_questions.json"),
}

def sa_answer_path(dataset_name: str, model_tag: str) -> Path:
    return Path(f"{dataset_name}_sa_qa/llm_answers/{dataset_name.lower()}_sa_answers_{model_tag}.jsonl")

In [5]:
def load_sa_answers(path: Path) -> list:
    """Load answer records from JSONL file."""
    records = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

def normalize_text(text: str) -> str:
    return " ".join(text.strip().split()).lower() if text else ""

In [6]:
def compute_token_f1(pred: str, gold: str) -> float:
    """Token-level F1 score: overlap of tokens between prediction and gold."""
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    
    pred_tokens = set(pred.split())
    gold_tokens = set(gold.split())
    
    if not pred_tokens or not gold_tokens:
        return 0.0
    
    intersection = len(pred_tokens & gold_tokens)
    precision = intersection / len(pred_tokens)
    recall = intersection / len(gold_tokens)
    
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


def compute_rouge_l(pred: str, gold: str) -> float:
    """ROUGE-L: Longest Common Subsequence based F1."""
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    
    pred_tokens = pred.split()
    gold_tokens = gold.split()
    
    # Longest Common Subsequence
    m, n = len(pred_tokens), len(gold_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_tokens[i-1] == gold_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    
    lcs_len = dp[m][n]
    precision = lcs_len / len(pred_tokens) if pred_tokens else 0.0
    recall = lcs_len / len(gold_tokens) if gold_tokens else 0.0
    
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

In [7]:
!pip install bert-score


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from bert_score import score

def compute_bertscore(pred: str, gold: str, lang: str = "en") -> float:
    """
    BERTScore: Semantic similarity using contextual embeddings.
    Returns F1 score (harmonic mean of precision and recall).
    """
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    
    try:
        # Compute BERTScore
        P, R, F1 = score([pred], [gold], lang=lang, verbose=False)
        return float(F1[0])
    except Exception as e:
        print(f"[WARN] BERTScore error: {e}")
        return 0.0

In [9]:
def compute_comparison_records(base_answers: list, compare_answers: list) -> list:
    """
    Compare answers from two models.
    base_answers: reference answers (llama3.1:8b)
    compare_answers: answers to compare (mistral)
    """
    # Create a map of base answers by qa_id
    base_map = {rec.get("qa_id"): rec for rec in base_answers}
    
    eval_records = []
    
    for rec in compare_answers:
        qa_id = rec.get("qa_id")
        if qa_id is None or qa_id not in base_map:
            continue
        
        base_rec = base_map[qa_id]
        
        # Get the base model answer (this is our "ground truth" for comparison)
        gold_raw = base_rec.get("llm_answer", "")
        # Get the compare model answer
        pred_raw = rec.get("llm_answer", "")
        
        pred = normalize_text(pred_raw)
        gold = normalize_text(gold_raw)
        
        # Compute metrics
        exact_match = 1.0 if pred == gold else 0.0
        token_f1 = compute_token_f1(pred, gold)
        rouge_l = compute_rouge_l(pred, gold)
        bertscore = compute_bertscore(pred_raw, gold_raw)  # Use raw text for BERTScore
        
        eval_records.append({
            "qa_id": qa_id,
            "base_answer": gold_raw,
            "compare_answer": pred_raw,
            "base_normalized": gold,
            "compare_normalized": pred,
            "exact_match": exact_match,
            "token_f1": token_f1,
            "rouge_l": rouge_l,
            "bertscore": bertscore,
        })
    
    return eval_records


def accuracy_from_records(records: list) -> dict:
    """Calculate Exact Match, Token F1, ROUGE-L, and BERTScore from records."""
    if not records:
        return {"exact_match": 0.0, "token_f1": 0.0, "rouge_l": 0.0, "bertscore": 0.0, "total": 0}
    
    total = len(records)
    exact_match = sum(r["exact_match"] for r in records) / total if total else 0.0
    token_f1 = sum(r["token_f1"] for r in records) / total if total else 0.0
    rouge_l = sum(r["rouge_l"] for r in records) / total if total else 0.0
    bertscore = sum(r["bertscore"] for r in records) / total if total else 0.0
    
    return {
        "exact_match": exact_match,
        "token_f1": token_f1,
        "rouge_l": rouge_l,
        "bertscore": bertscore,
        "total": total
    }

In [10]:
OUTPUT_DIR = Path("LLM_SA_Comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_csv = OUTPUT_DIR / f"Comparison_{BASE_TAG}_vs_{COMPARE_TAG}.csv"

print(f"[DEBUG] Current working directory: {Path.cwd()}")
print(f"[DEBUG] Base model: {BASE_MODEL} ({BASE_TAG})")
print(f"[DEBUG] Compare model: {COMPARE_MODEL} ({COMPARE_TAG})")
print(f"[DEBUG] Output CSV: {out_csv}")

header = ["Attack", "Total", "Exact Match", "Token F1", "ROUGE-L", "BERTScore"]
all_records = []

with out_csv.open("w", encoding="utf-8", newline="") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(header)

    for name in SA_DATASET_NAMES:
        base_path = sa_answer_path(name, BASE_TAG)
        compare_path = sa_answer_path(name, COMPARE_TAG)
        
        print(f"[DEBUG] Processing {name}:")
        print(f"  Base answers path: {base_path} -> exists: {base_path.exists()}")
        print(f"  Compare answers path: {compare_path} -> exists: {compare_path.exists()}")

        base_records = load_sa_answers(base_path)
        compare_records = load_sa_answers(compare_path)
        
        print(f"  Base answers loaded: {len(base_records)}")
        print(f"  Compare answers loaded: {len(compare_records)}")

        if not base_records:
            print(f"[WARN] Skip {name}: base ({BASE_TAG}) answers missing at {base_path}.")
            continue
        if not compare_records:
            print(f"[WARN] Skip {name}: compare ({COMPARE_TAG}) answers missing at {compare_path}.")
            continue

        print(f"[INFO] Comparing {name} ({BASE_TAG} vs {COMPARE_TAG})...")
        try:
            eval_records = compute_comparison_records(base_records, compare_records)
            print(f"  Comparison records created: {len(eval_records)}")
            
            if not eval_records:
                print(f"[ERROR] No comparison records generated for {name}")
                continue
            
            metrics = accuracy_from_records(eval_records)
            print(f"  Metrics: {metrics}")
            
            row = [name, metrics["total"]]
            row.extend([f"{metrics[m]:.3f}" for m in ["exact_match", "token_f1", "rouge_l", "bertscore"]])
            print(f"  Writing row: {row}")
            writer.writerow(row)

            all_records.extend(eval_records)
        except Exception as e:
            print(f"[ERROR] {name}: {e}")
            import traceback
            traceback.print_exc()
            continue

    if all_records:
        print(f"[DEBUG] Total records across all datasets: {len(all_records)}")
        metrics = accuracy_from_records(all_records)
        row = ["Combined", metrics["total"]]
        row.extend([f"{metrics[m]:.3f}" for m in ["exact_match", "token_f1", "rouge_l", "bertscore"]])
        writer.writerow(row)
    else:
        print(f"[WARN] No all_records to write combined metrics")

print(f"[INFO] Comparison results saved to {out_csv}")

[DEBUG] Current working directory: c:\Users\Dub\Documents\GitHub\CAN_QA\QA\Create_QA_ShortAnswer_Dataset
[DEBUG] Base model: llama3.1:8b (llama3_1_8b)
[DEBUG] Compare model: mistral:latest (mistral_latest)
[DEBUG] Output CSV: LLM_SA_Comparison\Comparison_llama3_1_8b_vs_mistral_latest.csv
[DEBUG] Processing DoS:
  Base answers path: DoS_sa_qa\llm_answers\dos_sa_answers_llama3_1_8b.jsonl -> exists: True
  Compare answers path: DoS_sa_qa\llm_answers\dos_sa_answers_mistral_latest.jsonl -> exists: True
  Base answers loaded: 916
  Compare answers loaded: 916
[INFO] Comparing DoS (llama3_1_8b vs mistral_latest)...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

  Comparison records created: 916
  Metrics: {'exact_match': 0.006550218340611353, 'token_f1': 0.15740053553918182, 'rouge_l': 0.15708212214762432, 'bertscore': 0.8637152988967938, 'total': 916}
  Writing row: ['DoS', 916, '0.007', '0.157', '0.157', '0.864']
[DEBUG] Processing Fuzzy:
  Base answers path: Fuzzy_sa_qa\llm_answers\fuzzy_sa_answers_llama3_1_8b.jsonl -> exists: True
  Compare answers path: Fuzzy_sa_qa\llm_answers\fuzzy_sa_answers_mistral_latest.jsonl -> exists: True
  Base answers loaded: 959
  Compare answers loaded: 959
[INFO] Comparing Fuzzy (llama3_1_8b vs mistral_latest)...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

  Comparison records created: 959
  Metrics: {'exact_match': 0.0072992700729927005, 'token_f1': 0.15700757358734418, 'rouge_l': 0.15699764260765303, 'bertscore': 0.8635396909539717, 'total': 959}
  Writing row: ['Fuzzy', 959, '0.007', '0.157', '0.157', '0.864']
[DEBUG] Processing Gear:
  Base answers path: Gear_sa_qa\llm_answers\gear_sa_answers_llama3_1_8b.jsonl -> exists: True
  Compare answers path: Gear_sa_qa\llm_answers\gear_sa_answers_mistral_latest.jsonl -> exists: True
  Base answers loaded: 1110
  Compare answers loaded: 1110
[INFO] Comparing Gear (llama3_1_8b vs mistral_latest)...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

  Comparison records created: 1110
  Metrics: {'exact_match': 0.002702702702702703, 'token_f1': 0.15148343648343648, 'rouge_l': 0.15133328633328633, 'bertscore': 0.8640526242084331, 'total': 1110}
  Writing row: ['Gear', 1110, '0.003', '0.151', '0.151', '0.864']
[DEBUG] Processing RPM:
  Base answers path: RPM_sa_qa\llm_answers\rpm_sa_answers_llama3_1_8b.jsonl -> exists: True
  Compare answers path: RPM_sa_qa\llm_answers\rpm_sa_answers_mistral_latest.jsonl -> exists: True
  Base answers loaded: 1155
  Compare answers loaded: 1155
[INFO] Comparing RPM (llama3_1_8b vs mistral_latest)...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

  Comparison records created: 1155
  Metrics: {'exact_match': 0.004329004329004329, 'token_f1': 0.15864658500258194, 'rouge_l': 0.1580304078595261, 'bertscore': 0.8630220003974386, 'total': 1155}
  Writing row: ['RPM', 1155, '0.004', '0.159', '0.158', '0.863']
[DEBUG] Total records across all datasets: 4140
[INFO] Comparison results saved to LLM_SA_Comparison\Comparison_llama3_1_8b_vs_mistral_latest.csv
